# CSI Sentinel â€” M3 on free Colab CPU
Run this notebook from a copy of the project in your Google Drive. It uses verified WiFall recordings, recording-safe holdouts and train-only normalization. These are research baselines, not validated alarms. Raw recordings and trained artifacts stay in your own runtime/Drive; never upload them to the cloud event API. Dataset license terms still need clarification before redistribution.


In [ ]:
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
PROJECT_ROOT = Path("/content/drive/MyDrive/Wifi_CSI")  # Copy this repository here first.
assert (PROJECT_ROOT / "pyproject.toml").exists(), "Set PROJECT_ROOT to your project copy"
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[ml]"], check=True)

In [ ]:
from urllib.request import urlretrieve

archive = Path(".cache/WiFall.zip")
archive.parent.mkdir(exist_ok=True)
if not archive.exists():
    urlretrieve("https://huggingface.co/datasets/RS2002/WiFall/resolve/main/WiFall.zip", archive)
with archive.open("rb") as handle:
    digest = hashlib.file_digest(handle, "sha256").hexdigest()
assert digest == "3a39d71aac6eaf8f37414c5d675323297e461b288bf4e70f8f42a9f6b94b5485", (
    "Dataset changed: inspect it before reusing the protocol"
)


def run(*args):
    subprocess.run([sys.executable, "-m", *args], check=True)


run("ml.prepare", "--archive", str(archive), "--dataset", "wifall", "--output", "data/wifall.npz")

## Choose a fixed protocol before looking at test results
Start with one fold to verify the setup. Remove `--only-fold ID0` for all ten participant holdouts. Each run creates a new directory; keep it to preserve the split assignments, artifacts and reports. CNN selection uses inner validation loss only. For leave-one-action-out use `--task fall --protocol loao`; the held-out-fall fold is explicitly skipped because no positive training class remains. Other LOAO test sets contain negatives only, so they measure false alarms, not fall recall.


In [ ]:
from datetime import UTC, datetime

output = "artifacts/colab-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
run(
    "ml.train",
    "--data",
    "data/wifall.npz",
    "--output",
    output,
    "--task",
    "action",
    "--protocol",
    "lopo",
    "--only-fold",
    "ID0",
)
print(json.dumps(json.loads((Path(output) / "summary.json").read_text()), indent=2))

In [ ]:
run(
    "ml.predict",
    "--model",
    str(Path(output) / "ID0/baseline.csimodel"),
    "--data",
    "data/wifall.npz",
    "--index",
    "0",
)
# Check ID0/splits.csv before treating a sample prediction as held-out evaluation.

## Interpret honestly
Windows inherit recording-level labels; WiFall does not provide precise fall-onset labels in these files. One participant contributes most of the data. Compare per-participant results and macro-F1 as well as pooled confusion matrices. Public-dataset models still need validation/retraining on your own hardware and room. Keep inference advisory; the M2 stillness state machine remains separate.
